# Lab 10: Word Embeddings: Training Word2Vec (Skip-Gram with Negative Sampling) & PyTorch `nn.Embedding`

Welcome to Laboratory 10! In this lab, we learn how dense semantic vector spaces are learned from raw text:
1. **The Distributional Hypothesis**: *"A word is characterized by the company it keeps"* (Firth, 1957).
2. **Skip-Gram with Negative Sampling (SGNS)**: Formulate word embedding learning as binary logistic classification between true context pairs and noise pairs.
3. **PyTorch `nn.Embedding`**: Implement embedding lookup matrices and vector similarity metrics (Cosine Similarity).


## 1. Technical Preliminaries & Environment Setup


In [ ]:
# Import PyTorch and mathematical computing libraries
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Set deterministic random seed
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active Device:', device)


## 2. Skip-Gram with Negative Sampling (SGNS) Architecture

### Architecture & Mathematical Formulation: `SkipGram`
Given a target center word $w_t$, positive context word $w_c$, and $K$ negative noise words $\{w_{n_1}, \dots, w_{n_K}\}$:
* **Target Embedding Lookup**: $\mathbf{u} = \mathbf{U}[w_t] \in \mathbb{R}^{D}$
* **Positive Context Embedding Lookup**: $\mathbf{v}_{pos} = \mathbf{V}[w_c] \in \mathbb{R}^{D}$
* **Negative Samples Embedding Lookup**: $\mathbf{V}_{neg} \in \mathbb{R}^{K \times D}$
* **Objective Function (Negative Sampling Loss)**:
  $$\mathcal{L}_{SGNS} = -\log \sigma(\mathbf{u}^T \mathbf{v}_{pos}) - \sum_{k=1}^{K} \log \sigma(-\mathbf{u}^T \mathbf{v}_{n_k})$$

This formulation replaces computationally expensive softmax normalization over the full vocabulary ($|V|$) with efficient binary cross-entropy calculations over $K+1$ vectors.


In [ ]:
# Define the Word2Vec Skip-Gram with Negative Sampling Neural Architecture
class SkipGram(nn.Module):
    """Word2Vec Skip-Gram Architecture with Target and Context Embedding Tables."""
    def __init__(self, vocab_size: int, embed_dim: int = 16):
        super(SkipGram, self).__init__()
        # Target (Center) word embedding matrix U of shape (Vocab_Size, Embed_Dim)
        self.u_embed = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        
        # Context word embedding matrix V of shape (Vocab_Size, Embed_Dim)
        self.v_embed = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        
        # Initialize embeddings with uniform random values
        self.u_embed.weight.data.uniform_(-0.5 / embed_dim, 0.5 / embed_dim)
        self.v_embed.weight.data.uniform_(-0.5 / embed_dim, 0.5 / embed_dim)
        
    def forward(self, target: torch.Tensor, pos_ctx: torch.Tensor, neg_ctx: torch.Tensor) -> torch.Tensor:
        """
        Args:
            target: Center word indices of shape (Batch_Size,)
            pos_ctx: True positive context word indices of shape (Batch_Size,)
            neg_ctx: Randomly sampled negative word indices of shape (Batch_Size, K_neg)
        Returns:
            Scalar SGNS loss averaged across the mini-batch.
        """
        # Step 1: Look up embeddings
        u = self.u_embed(target)        # Shape: (B, D)
        v_pos = self.v_embed(pos_ctx)   # Shape: (B, D)
        v_neg = self.v_embed(neg_ctx)   # Shape: (B, K_neg, D)
        
        # Step 2: Positive Pair Loss: -log(sigmoid(u . v_pos))
        pos_dot = (u * v_pos).sum(dim=1)                                           # Shape: (B,)
        pos_loss = -torch.log(torch.sigmoid(pos_dot) + 1e-7)                       # Shape: (B,)
        
        # Step 3: Negative Pair Loss: -sum(log(sigmoid(-u . v_neg))) using Batched Matrix Multiply (BMM)
        neg_dot = torch.bmm(v_neg, u.unsqueeze(2)).squeeze(2)                      # Shape: (B, K_neg)
        neg_loss = -torch.log(torch.sigmoid(-neg_dot) + 1e-7).sum(dim=1)           # Shape: (B,)
        
        # Step 4: Combine positive and negative losses
        total_loss = (pos_loss + neg_loss).mean()
        return total_loss

# Instantiate SkipGram model with vocabulary of 50 tokens and 16 embedding dimensions
sg_model = SkipGram(vocab_size=50, embed_dim=16).to(device)
print('Initialized SkipGram Word2Vec Model:\n', sg_model)


### Embedding Vector Analysis: Cosine Similarity
The function below computes semantic similarity between word vectors in the learned continuous geometric space:
$$\text{Cosine Similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2} \in [-1, 1]$$


In [ ]:
def compute_cosine_similarity(vec1: torch.Tensor, vec2: torch.Tensor) -> float:
    """Calculates normalized Cosine Similarity between two embedding vectors."""
    dot_product = torch.dot(vec1, vec2)
    norm1 = torch.norm(vec1)
    norm2 = torch.norm(vec2)
    similarity = dot_product / (norm1 * norm2 + 1e-8)
    return similarity.item()

# Sample vector cosine similarity test
v1 = torch.tensor([1.0, 2.0, 3.0])
v2 = torch.tensor([1.0, 2.0, 2.8])
print('Sample Vector Cosine Similarity:', round(compute_cosine_similarity(v1, v2), 4))


## 3. Summary & Key Takeaways
1. **Continuous Vector Representations**: Words are mapped to points in dense vector space where spatial proximity reflects semantic similarity.
2. **Negative Sampling**: Converts expensive full-vocabulary softmax into scalable binary logistic regressions.
3. **Linear Substructures**: Word embeddings exhibit emergent algebraic properties (e.g., $\mathbf{v}_{king} - \mathbf{v}_{man} + \mathbf{v}_{woman} \approx \mathbf{v}_{queen}$).
